In [1]:
%%writefile gsec_labels.py
import pandas as pd


def spreads_from_table(table):
    return pd.DataFrame({
        "2s5s": (table[5] - table[2]) * 100,
        "2s10s": (table[10] - table[2]) * 100,
        "5s10s": (table[10] - table[5]) * 100,
        "10s30s": (table[30] - table[10]) * 100,
    })


def classify(change_bps, threshold):
    """Steepening if the spread widened by at least the threshold,
    flattening if it narrowed by at least the threshold, else little changed."""
    if abs(change_bps) < threshold:
        return "little changed"
    return "steepening" if change_bps > 0 else "flattening"


def level_label(spread_value):
    return "inverted" if spread_value < 0 else "upward sloping"


def noise_floor(spreads, n_days=10):
    """Median absolute day-to-day change over the last n_days consecutive rows."""
    return spreads.tail(n_days).diff().abs().median()


def label_period(spreads, start, end, threshold):
    rows = []
    for name in spreads.columns:
        change = spreads.loc[end, name] - spreads.loc[start, name]
        rows.append({
            "spread": name,
            "start": round(spreads.loc[start, name], 1),
            "end": round(spreads.loc[end, name], 1),
            "change_bps": round(change, 1),
            "label": classify(change, threshold[name]),
            "shape_now": level_label(spreads.loc[end, name]),
        })
    return pd.DataFrame(rows)

Writing gsec_labels.py


In [6]:
%%writefile test_gsec_labels.py
import pandas as pd
from gsec_labels import classify, level_label, spreads_from_table, noise_floor, label_period


def test_spreads_from_table():
    table = pd.DataFrame(
        [[6.0, 6.5, 7.0, 7.5], [6.0, 6.4, 7.2, 7.4]],
        index=pd.to_datetime(["2026-01-01", "2026-01-02"]),
        columns=[2, 5, 10, 30],
    )
    s = spreads_from_table(table)
    assert abs(s.loc["2026-01-01", "2s5s"] - 50) < 1e-9
    assert abs(s.loc["2026-01-01", "2s10s"] - 100) < 1e-9
    assert abs(s.loc["2026-01-02", "5s10s"] - 80) < 1e-9
    assert abs(s.loc["2026-01-02", "10s30s"] - 20) < 1e-9


def test_classify_directions():
    assert classify(5, 2) == "steepening"
    assert classify(-5, 2) == "flattening"
    assert classify(1, 2) == "little changed"


def test_classify_boundary():
    assert classify(2, 2) == "steepening"
    assert classify(-2, 2) == "flattening"
    assert classify(1.99, 2) == "little changed"


def test_inversion_label():
    assert level_label(-1) == "inverted"
    assert level_label(0.5) == "upward sloping"
    assert level_label(0) == "upward sloping"


def test_noise_floor():
    s = pd.DataFrame({"x": [0, 2, 1, 4]})
    assert noise_floor(s, n_days=4)["x"] == 2


def test_label_period():
    s = pd.DataFrame({"2s10s": [100.0, 120.0]},
                     index=pd.to_datetime(["2026-01-01", "2026-01-02"]))
    out = label_period(s, "2026-01-01", "2026-01-02", pd.Series({"2s10s": 5}))
    assert out.loc[0, "change_bps"] == 20.0
    assert out.loc[0, "label"] == "steepening"
    assert out.loc[0, "shape_now"] == "upward sloping"

Writing test_gsec_labels.py


In [3]:
import pandas as pd
from gsec_data import build_table
from gsec_labels import *

table = build_table("data")
spreads = spreads_from_table(table)
thr = noise_floor(spreads, 10)
print("noise floor (bps):")
print(thr.round(1))

noise floor (bps):
2s5s      6.9
2s10s     2.5
5s10s     3.2
10s30s    1.9
dtype: float64


In [4]:
windows = {
    "daily (15 Oct to 16 Oct)": ("2026-10-15", "2026-10-16"),
    "weekly (9 Oct to 16 Oct)": ("2026-10-09", "2026-10-16"),
    "about a month (18 Sep to 16 Oct)": ("2026-09-18", "2026-10-16"),
}
for title, (a, b) in windows.items():
    print(title)
    display(label_period(spreads, a, b, thr))

daily (15 Oct to 16 Oct)


,spread,start,end,change_bps,label,shape_now
0,2s5s,63.1,59.8,-3.2,little changed,upward sloping
1,2s10s,78.0,76.0,-1.9,little changed,upward sloping
2,5s10s,14.9,16.2,1.3,little changed,upward sloping
3,10s30s,36.4,36.8,0.4,little changed,upward sloping


weekly (9 Oct to 16 Oct)


,spread,start,end,change_bps,label,shape_now
0,2s5s,49.9,59.8,10.0,steepening,upward sloping
1,2s10s,81.4,76.0,-5.4,flattening,upward sloping
2,5s10s,31.6,16.2,-15.4,flattening,upward sloping
3,10s30s,33.6,36.8,3.2,steepening,upward sloping


about a month (18 Sep to 16 Oct)


,spread,start,end,change_bps,label,shape_now
0,2s5s,51.7,59.8,8.1,steepening,upward sloping
1,2s10s,73.5,76.0,2.6,steepening,upward sloping
2,5s10s,21.8,16.2,-5.6,flattening,upward sloping
3,10s30s,50.8,36.8,-14.0,flattening,upward sloping


In [5]:
print("lowest value of each spread over all dates:")
print(spreads.min().round(1))
print("number of dates with a negative spread:", (spreads < 0).sum().to_dict())

lowest value of each spread over all dates:
2s5s      26.5
2s10s     51.7
5s10s     14.9
10s30s    33.6
dtype: float64
number of dates with a negative spread: {'2s5s': 0, '2s10s': 0, '5s10s': 0, '10s30s': 0}
